In [1]:
import torch
from faster_whisper import WhisperModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

print("Loading Whisper model...")
whisper_model = WhisperModel("medium", device="cuda", compute_type="float16")
print("Whisper loaded!")

audio_file = "../audio/test3.m4a"

print(f"\nTranscribing {audio_file}...")
segments, info = whisper_model.transcribe(
    audio_file,
    beam_size=5,
    language="hi",
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500)
)

print(f"Detected language: {info.language} (confidence: {info.language_probability:.2f})")
print("\n--- Transcription ---")

full_text = ""
for segment in segments:
    print(f"[{segment.start:.1f}s → {segment.end:.1f}s] {segment.text}")
    full_text += segment.text + " "

full_text = full_text.strip()
print(f"\nFull text:\n{full_text}")

print("\nLoading IndicTrans2 model (CPU)... this may take a minute first time")

model_name = "ai4bharat/indictrans2-indic-en-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float32 
).to("cpu")

ip = IndicProcessor(inference=True)

print("Translating Hindi → English...")

batch = ip.preprocess_batch(
    [full_text],
    src_lang="hin_Deva",   
    tgt_lang="eng_Latn"    
)

inputs = tokenizer(
    batch,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    return_attention_mask=True
).to("cpu")

with torch.no_grad():
    generated_tokens = trans_model.generate(
        **inputs,
        use_cache=True,
        min_length=0,
        max_length=256,
        num_beams=5,
        num_return_sequences=1
    )

with tokenizer.as_target_tokenizer():
    generated_tokens = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

translations = ip.postprocess_batch(generated_tokens, lang="eng_Latn")

print("\n--- Translation (English) ---")
print(translations[0])

Loading Whisper model...
Whisper loaded!

Transcribing ../audio/test3.m4a...
Detected language: hi (confidence: 1.00)

--- Transcription ---
[0.4s → 4.0s]  मुझे क्रिषी विग्यान के बारे में और जानकारी चाहिए है।

Full text:
मुझे क्रिषी विग्यान के बारे में और जानकारी चाहिए है।

Loading IndicTrans2 model (CPU)... this may take a minute first time


C:\Users\naman\Downloads\Projects\MulitLingualRAG\raprod\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\naman\Downloads\Projects\MulitLingualRAG\raprod\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Translating Hindi → English...

--- Translation (English) ---
I need more information about Krishi Vigyan.


C:\Users\naman\Downloads\Projects\MulitLingualRAG\raprod\Lib\site-packages\transformers\tokenization_utils_base.py:3921: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [2]:
import sys 
sys.path.append("C:/Users/naman/Downloads/Projects/MulitLingualRAG/src")
from cleaner import TextCleaner
cleaner = TextCleaner()
raw = translations[0]
clean= cleaner.clean(raw)
print("Raw : " ,raw)
print("Clean : ",clean)

Raw :  I need more information about Krishi Vigyan.
Clean :  ('I need more information about Krishi Vigyan.',)


In [4]:
import sys
sys.path.append("C:/Users/naman/Downloads/Projects/MulitLingualRAG/src")
from router import IntentRouter 
router = IntentRouter()
queries = [
    "What documents do I need for Kisan Credit Card?",
    "I want to apply for a new ration card",
    "What are the rules under PM Kisan scheme?",
    "Register me for PM Kisan Yojana"
]
for q in queries :
    result = router.route(q)
    print(result)

Intent : asking a question about rules, documents, or eligibility , Score : 0.88
rag
None
Intent : applying, registering, submitting, or filling a form , Score : 0.67
apply
None
Intent : asking a question about rules, documents, or eligibility , Score : 0.86
rag
None
Intent : applying, registering, submitting, or filling a form , Score : 0.67
apply
None


In [7]:
import sys
sys.path.append("C:/Users/naman/Downloads/Projects/MulitLingualRAG")

from parser import DocumentParser

parser = DocumentParser(poppler_path=r"C:\poppler\Library\bin")

pages = parser.parse("../docs/NA_CV.pdf")

for chunk in pages:
    print(f"Page {chunk['page']} ({chunk['type']}):")
    print(chunk['text'][:300])
    print("---")


Parsing: NA_CV.pdf (1 pages)
  Page 1: digital
  Done — 2 content chunks extracted



KeyError: 'page_num'

In [8]:
print("------")
import sys
sys.path.append("C:/Users/naman/Downloads/Projects/MulitLingualRAG/src")
print("------")
from rag import RAGPipeline
print("-----")
rag = RAGPipeline(docs_folder="../docs")

# First time — ingest your PDFs
rag.ingest()

# Test retrieval
results = rag.retrieve("What documents are needed for Kisan Credit Card?")

for r in results:
    print(f"Score: {r['score']} | Source: {r['source']} | Page: {r['page']}")
    print(r['text'][:300])
    print("---")

------
------
-----
Loading embedding model...
RAG pipeline ready!
Found 2 PDFs: ['NA_CV.pdf', 'RAG_Test_Document_No_Questions.pdf']

Parsing: NA_CV.pdf (1 pages)
  Page 1: digital
  Done — 2 content chunks extracted



KeyError: 'page_num'